In [2]:
pip install ta

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29421 sha256=2406bc1be5a88daaa141e380bf406ae0f59ab2d68219bcd848e289384969799a
  Stored in directory: c:\users\xizhenh\appdata\local\pip\cache\wheels\5c\a1\5f\c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd 
import numpy as np
import ta

def add_features(df):
    df = df.copy()
    df.sort_index(inplace=True)

    # Daily return
    df['return_1d'] = df['Close'].pct_change()
    df['return_3d'] = df['Close'].pct_change(3)
    df['return_5d'] = df['Close'].pct_change(5)
    df['cumulative_return_10d'] = df['Close'].pct_change(10)

    # Technical indicators
    df['sma_5'] = ta.trend.sma_indicator(df['Close'], window=5)
    df['sma_10'] = ta.trend.sma_indicator(df['Close'], window=10)
    df['ema_12'] = ta.trend.ema_indicator(df['Close'], window=12)
    df['ema_26'] = ta.trend.ema_indicator(df['Close'], window=26)
    df['rsi'] = ta.momentum.rsi(df['Close'], window=14)
    df['macd'] = ta.trend.macd_diff(df['Close'])

    # Bollinger Bands
    bb = ta.volatility.BollingerBands(close=df['Close'], window=20, window_dev=2)
    df['bollinger_h'] = bb.bollinger_hband()
    df['bollinger_l'] = bb.bollinger_lband()
    df['bollinger_width'] = bb.bollinger_wband()

    # Volatility
    df['rolling_std_5'] = df['return_1d'].rolling(5).std()
    df['rolling_std_10'] = df['return_1d'].rolling(10).std()

    # Volume-related features
    df['volume_change'] = df['Volume'].pct_change()
    df['avg_volume_5d'] = df['Volume'].rolling(5).mean()

    # Price vs moving averages
    df['price_above_sma_10'] = df['Close'] - df['sma_10']
    df['sma_ratio_5_10'] = df['sma_5'] / df['sma_10']

    # Target variables (future returns)
    for i in range(1, 11):
        df[f'target_price_{i}d'] = df['Close'].shift(-i)

    df.dropna(inplace=True)
    return df

In [10]:
if __name__ == "__main__":
    # Load raw data
    df = pd.read_csv("C:\\Users\\xizhenh\\Desktop\\Hunter\\Trading System\\data\\raw_data.csv", parse_dates=True, index_col="Date")
    
    # Apply feature engineering to each ticker separately
    final_df = df.groupby("ticker").apply(add_features).reset_index(drop=True)

    # Save to file
    final_df.to_csv("C:\\Users\\xizhenh\\Desktop\\Hunter\\Trading System\\data\\processed_data.csv", index=False)
    print("Features and target created and saved.")

ValueError: 'Date' is not in list